In [1]:
import pandas as pd
import os
from pathlib import Path

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [3]:
parquet_path = Path("GEBCO/merged_all_gebco.parquet")
output_path = Path("Final/column_coverage.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(parquet_path)

column_coverage = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "non_null_count": [df[col].notna().sum() for col in df.columns],
    "total_count": len(df),
})

column_coverage["coverage_pct"] = (
    column_coverage["non_null_count"] / column_coverage["total_count"] * 100
)

column_coverage.to_csv(output_path, index=False)
print(len(df))
print(f"Saved: {output_path}")
print(column_coverage)

2365237
Saved: Final\column_coverage.csv
                    column    dtype  non_null_count  total_count  coverage_pct
0                       ID    Int64               0      2365237      0.000000
1   acceptedScientificName   string         2365237      2365237    100.000000
2                  aphiaID    Int64         2365237      2365237    100.000000
3              areaSampled  float64          293031      2365237     12.389076
4          areaSampledUnit   string          296645      2365237     12.541872
..                     ...      ...             ...          ...           ...
64                    Keep      str         2023158      2365237     85.537221
65         acceptedAphiaID  float64         2365237      2365237    100.000000
66                isMarine  boolean         2365026      2365237     99.991079
67              isBrackish  boolean         1929561      2365237     81.580028
68      trawlingDistance_m  float64             934      2365237      0.039489

[69 rows x

In [4]:
columns_to_drop = ["percentCoverage", "Keep", "Keep_x", "Keep_y"]

final_df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
final_output_path = Path("Final/merged_all_final.parquet")
final_output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_parquet(final_output_path, index=False)
print(f"Saved: {final_output_path}")
print(final_df.shape)

Saved: Final\merged_all_final.parquet
(2365237, 65)


In [5]:
# Update the 'flagged' column: set to True where 'flaggedComment' is not null
final_df['flagged'] = final_df['flagged'] | final_df['flaggedComment'].notna()

# Save the updated dataframe
final_df.to_parquet(final_output_path, index=False)
print(f"Saved: {final_output_path}")

Saved: Final\merged_all_final.parquet


In [10]:
# Check for mojibake (encoding issues) in string columns and repair recoverable text
mojibake_patterns = [
    "�",      # replacement character
    "â€",     # common UTF-8/Windows-1252 mojibake prefix
    "â€™",
    "â€œ",
    "â€�",
    "â€ž",
    "ï¿½",     # replacement char rendered as mojibake
    "Ã",
    "Â",
]

# Try to recover strings that were decoded with the wrong encoding
def fix_mojibake_text(value):
    if pd.isna(value) or not isinstance(value, str):
        return value

    for source_encoding in ("cp1252", "latin1"):
        try:
            fixed = value.encode(source_encoding).decode("utf-8")
            if fixed != value:
                return fixed
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass

    return value

# Check string columns for common mojibake indicators
string_cols = final_df.select_dtypes(include=['string', 'object']).columns
repaired_df = final_df.copy()
unrecoverable_examples = []

for col in string_cols:
    # Use Python-backed strings to avoid Arrow regex limitations
    series = repaired_df[col].astype('string[python]')

    mojibake_mask = pd.Series(False, index=series.index)
    for pattern in mojibake_patterns:
        mojibake_mask = mojibake_mask | series.str.contains(pattern, na=False, regex=False)

    if mojibake_mask.any():
        sample_original = repaired_df.loc[mojibake_mask, col].drop_duplicates().head(10)
        sample_corrected = sample_original.map(fix_mojibake_text)
        preview_df = pd.DataFrame({
            "original": sample_original.values,
            "corrected": sample_corrected.values,
        })
        preview_df = preview_df[preview_df["original"] != preview_df["corrected"]]

        if not preview_df.empty:
            print(f"\n--- Potential mojibake in '{col}' ({mojibake_mask.sum()} rows) ---")
            print(preview_df.to_string(index=False))

        unrecoverable_mask = sample_original == sample_corrected
        if unrecoverable_mask.any():
            unrecoverable_df = pd.DataFrame({
                "column": col,
                "value": sample_original[unrecoverable_mask].values,
            })
            unrecoverable_examples.append(unrecoverable_df)

    repaired_df[col] = repaired_df[col].map(fix_mojibake_text)

repaired_output_path = Path("Final/merged_all_final_repaired.parquet")
repaired_output_path.parent.mkdir(parents=True, exist_ok=True)
repaired_df.to_parquet(repaired_output_path, index=False)
print(f"\nSaved repaired parquet: {repaired_output_path}")

if unrecoverable_examples:
    unrecoverable_df = pd.concat(unrecoverable_examples, ignore_index=True).drop_duplicates()
    print("\n--- Potential entries not recovered by the repair pass ---")
    print(unrecoverable_df.head(20).to_string(index=False))
else:
    print("\nNo unrecoverable mojibake-like entries were found in the sampled values.")

print("\nMojibake repair complete!")


--- Potential mojibake in 'eventID' (508 rows) ---
           original         corrected
  8428.03-MÃ¥rs 2-2  8428.03-Mårs 2-2
  8428.03-MÃ¥rs 2-1  8428.03-Mårs 2-1
7362-GislÃ¸yaÂ  3-1 7362-Gisløya  3-1
      8203-Ã…rs 3-1      8203-Års 3-1
      8203-Ã…rs 3-2      8203-Års 3-2
 8429.02-GrÃ¸tS 1-1 8429.02-GrøtS 1-1
 8429.02-GrÃ¸tS 1-2 8429.02-GrøtS 1-2
 8429.01-GrÃ¸tN 1-1 8429.01-GrøtN 1-1
     8114-SkÃ¥l 3-1     8114-Skål 3-1
7362-GislÃ¸yaÂ  3-2 7362-Gisløya  3-2

--- Potential mojibake in 'eventRemarks' (3045 rows) ---
                                                                   original                                                                  corrected
Used in JordÃ -Molina et al. 2023; respiration incubations prior to sieving Used in Jordà-Molina et al. 2023; respiration incubations prior to sieving
                                          Used in JordÃ -Molina et al. 2023                                           Used in Jordà-Molina et al. 2023
                   

In [14]:
stations = pd.read_parquet("Final/merged_all_final.parquet")

cols = [
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "datasetKey",
    "datasetTitle",
    "sourceDatabase",
    "year",
    "gebcoDepth",
    "startDayOfYear",
]
all_stations = stations[cols].drop_duplicates()

total_stations = len(all_stations)
print(f"Total unique stations: {total_stations}")

all_stations.to_parquet("Final/all_stations.parquet", index=False)

all_stations

Total unique stations: 237205


,decimalLatitudeStart,decimalLongitudeStart,datasetKey,datasetTitle,sourceDatabase,year,gebcoDepth,startDayOfYear
0,73.833333,25.891500,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2016,-457.0,274
1,68.184360,10.338713,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2012,-875.0,123
2,67.035000,10.991333,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2012,-243.0,131
3,70.602667,20.689000,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2010,-118.0,264
4,70.245000,31.055000,99006141-bf02-4821-8e6e-5818a1cbf70a,imr_mareano_beamtrawl,GBIF,2014,-167.0,106
...,...,...,...,...,...,...,...,...
2365212,68.906255,14.356099,37e512e6-732f-49fb-a4bd-544522b42b25,Lophelia pertusa reefs in Norwegian seawaters,OBIS,<NA>,-238.0,<NA>
2365223,68.931869,14.381949,e5389590-6d7e-4752-b84b-faa2a411db87,ICES Vulnerable Marine Ecosystems,OBIS,<NA>,-245.0,<NA>
2365224,68.071999,39.521999,a145f369-b272-46f7-86bc-08412b1ee927,Arctic benthic invertebrate collection of the ...,OBIS,1876,2.0,193
2365232,73.500000,131.673004,a145f369-b272-46f7-86bc-08412b1ee927,Arctic benthic invertebrate collection of the ...,OBIS,1993,-21.0,232


In [ ]:
# Count unique aphiaID per station row and overwrite Final/all_stations.parquet

count_col = "unique_aphiaID_count"

aphia_counts = (
    stations.groupby(cols, dropna=False)["aphiaID"]
    .nunique()
    .reset_index(name=count_col)
)

if count_col in all_stations.columns:
    all_stations = all_stations.drop(columns=[count_col])

all_stations = all_stations.merge(aphia_counts, on=cols, how="left")
all_stations[count_col] = all_stations[count_col].fillna(0).astype("Int64")

all_stations.to_parquet("Final/all_stations.parquet", index=False)

print(f"Saved: Final/all_stations.parquet")
print(all_stations[[*cols, count_col]].head())

In [13]:
datasets_path = Path("Final/datasets.csv")
datasets_path.parent.mkdir(parents=True, exist_ok=True)

datasets_df = (
    pd.read_parquet("Final/merged_all_final_repaired.parquet", columns=["datasetTitle"])
    .dropna()
    .drop_duplicates()
    .sort_values("datasetTitle")
)

datasets_df.to_csv(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(datasets_df.shape)
print(datasets_df.head())

Saved: Final\datasets.csv
(434, 1)
                                              datasetTitle
2290744  1776-1997 Paul F. Clark North East Atlantic Cr...
2324801  A Biotic Database of Indo-Pacific Marine Mollusks
906610   A new deepwater species of Calliopiidae, Halir...
1785196  A new genus and five new species of the subfam...
1734954  A revision of the genus Calcigorgia (Cnidaria,...


## If data types are not correct

In [ ]:
schema_path = Path("column_data_types.xlsx")
schema_df = pd.read_excel(schema_path, sheet_name="types")
schema_map = dict(zip(schema_df["column"], schema_df["final"]))


def coerce_column_to_schema(series, dtype_name):
    dtype_name = str(dtype_name).strip()

    if dtype_name == "string":
        return series.astype("string")
    if dtype_name == "float64":
        return pd.to_numeric(series, errors="coerce")
    if dtype_name == "Int64":
        return pd.to_numeric(series, errors="coerce").round().astype("Int64")
    if dtype_name == "boolean":
        normalized = series.astype("string").str.strip().str.lower()
        return normalized.map({"true": True, "false": False, "1": True, "0": False}).astype("boolean")

    return series


def enforce_schema_in_file(parquet_path):
    parquet_path = Path(parquet_path)
    if not parquet_path.exists():
        print(f"Skipped missing file: {parquet_path}")
        return

    frame = pd.read_parquet(parquet_path)

    for column_name, dtype_name in schema_map.items():
        if column_name in frame.columns:
            frame[column_name] = coerce_column_to_schema(frame[column_name], dtype_name)

    ordered_columns = [column for column in schema_df["column"] if column in frame.columns]
    ordered_columns.extend([column for column in frame.columns if column not in ordered_columns])
    frame = frame[ordered_columns]

    frame.to_parquet(parquet_path, index=False)
    print(f"Enforced schema for {parquet_path}")


for parquet_file in [
    "All_data/merged_all_mof.parquet",
    "All_data/merged_all_filtered.parquet",
]:
    enforce_schema_in_file(parquet_file)